# SAM 3 Auto-Labeling for Building Defect Detection

This notebook uses Meta's SAM 3 (Segment Anything Model 3) to automatically label 4,193 filtered images from Dataset 6.

**Goal**: Recover 2,000-3,000 usable images with text-prompted segmentation.

**Requirements**:
- Google Colab with GPU runtime (T4 or better)
- Hugging Face account with SAM 3 checkpoint access
- Upload `filtered_images.zip` to Google Drive

**Estimated Time**: 2-4 hours for 4,193 images

## 1. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# Install SAM 3 from GitHub
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e .
!pip install opencv-python pillow tqdm
%cd ..

print("✅ SAM 3 installed")

In [ ]:
# Authenticate with Hugging Face
from huggingface_hub import notebook_login
notebook_login()

print("✅ Hugging Face authenticated")
print("⚠️  Make sure you have access to facebook/sam3 model on Hugging Face")

## 2. Load SAM 3 Model

In [ ]:
import torch
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
import numpy as np
from PIL import Image
import cv2
import os
from tqdm.auto import tqdm
import json
import zipfile

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load SAM 3 model
print("Loading SAM 3 model from Hugging Face...")
model = build_sam3_image_model(
    device=device,
    load_from_HF=True
)
processor = Sam3Processor(model)

print("✅ SAM 3 model loaded successfully")

## 3. Define Defect Classes and Prompts

In [ ]:
# 15 target defect classes with text prompts
DEFECT_PROMPTS = {
    0: ['structural crack', 'crack in wall', 'crack in concrete', 'crack', 'wall crack'],
    1: ['water damage', 'water stain', 'water leak', 'damp', 'dampness', 'moisture damage'],
    2: ['mold', 'mould', 'fungus', 'mildew', 'black mold'],
    3: ['rust', 'corrosion', 'rusty metal', 'corroded surface'],
    4: ['peeling paint', 'flaking paint', 'paint damage', 'paint peeling off'],
    5: ['efflorescence', 'white deposit', 'salt deposit', 'mineral stain'],
    6: ['spalling', 'concrete spalling', 'surface spalling', 'concrete damage'],
    7: ['deterioration', 'material deterioration', 'surface deterioration'],
    8: ['structural damage', 'broken structure', 'damaged structure'],
    9: ['settlement', 'foundation settlement', 'sinking', 'subsidence'],
    10: ['delamination', 'surface delamination', 'layer separation'],
    11: ['discoloration', 'staining', 'color change', 'surface stain'],
    12: ['biological growth', 'algae', 'moss', 'plant growth'],
    13: ['honeycomb', 'concrete honeycomb', 'porous concrete'],
    14: ['missing grout', 'missing mortar', 'grout loss']
}

CLASS_NAMES = [
    'crack', 'water_damage', 'mold', 'rust', 'peeling_paint',
    'efflorescence', 'spalling', 'deterioration', 'structural_damage',
    'settlement', 'delamination', 'discoloration', 'biological_growth',
    'honeycomb', 'missing_grout'
]

# Quality thresholds
MIN_CONFIDENCE = 0.50  # SAM 3 confidence threshold
MIN_AREA = 0.001      # Minimum area (0.1% of image)
MAX_AREA = 0.75       # Maximum area (75% of image)
MAX_INSTANCES_PER_CLASS = 20  # Max instances per defect type

print(f"✅ Configured {len(DEFECT_PROMPTS)} defect classes")

## 4. Extract Filtered Images

In [ ]:
# Extract filtered images from ZIP
# UPLOAD filtered_images.zip to your Google Drive first
ZIP_PATH = '/content/drive/MyDrive/filtered_images.zip'  # Update this path
EXTRACT_DIR = '/content/filtered_images'

print(f"Extracting images from {ZIP_PATH}...")
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Get all image files
image_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files.append(os.path.join(root, file))

print(f"✅ Found {len(image_files)} images to process")

## 5. SAM 3 Auto-Labeling Function

In [ ]:
def segment_with_sam3(image_path, text_prompt, confidence_threshold=0.5):
    """
    Use SAM 3 text prompt to segment defects
    Returns list of bounding boxes and masks
    """
    try:
        # Load image
        image = Image.open(image_path).convert('RGB')
        image_np = np.array(image)
        height, width = image_np.shape[:2]
        
        # Set image in processor
        inference_state = processor.set_image(image_np)
        
        # Run text-prompted segmentation
        output = processor.set_text_prompt(
            state=inference_state,
            prompt=text_prompt
        )
        
        # Extract masks and scores
        masks = output.get('masks', [])
        scores = output.get('scores', [])
        
        # Filter by confidence and convert to bounding boxes
        detections = []
        for mask, score in zip(masks, scores):
            if score < confidence_threshold:
                continue
            
            # Convert mask to bounding box
            mask_binary = (mask > 0.5).astype(np.uint8)
            contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                x, y, w, h = cv2.boundingRect(contour)
                area = w * h / (width * height)
                
                # Filter by area
                if area < MIN_AREA or area > MAX_AREA:
                    continue
                
                detections.append({
                    'bbox': [x, y, w, h],
                    'confidence': float(score),
                    'mask': mask_binary
                })
        
        return detections
    
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return []

print("✅ SAM 3 segmentation function ready")

## 6. Process All Images with Progress Tracking

In [ ]:
# Create output directories
OUTPUT_DIR = '/content/sam3_labels'
LABELS_DIR = os.path.join(OUTPUT_DIR, 'labels')
STATS_FILE = os.path.join(OUTPUT_DIR, 'labeling_stats.json')

os.makedirs(LABELS_DIR, exist_ok=True)

# Statistics tracking
stats = {
    'total_images': len(image_files),
    'processed': 0,
    'images_with_defects': 0,
    'images_without_defects': 0,
    'total_detections': 0,
    'detections_per_class': {name: 0 for name in CLASS_NAMES},
    'failed_images': []
}

# Process images
print(f"\n🚀 Starting auto-labeling of {len(image_files)} images...\n")

for image_path in tqdm(image_files, desc="Auto-labeling"):
    try:
        image_name = os.path.basename(image_path)
        image_id = os.path.splitext(image_name)[0]
        
        # Load image to get dimensions
        image = Image.open(image_path)
        width, height = image.size
        
        # Try each defect class
        all_detections = []
        
        for class_id, prompts in DEFECT_PROMPTS.items():
            class_detections = []
            
            # Try each prompt for this class (exhaustive search)
            for prompt in prompts:
                detections = segment_with_sam3(image_path, prompt, MIN_CONFIDENCE)
                class_detections.extend(detections)
                
                # Stop if we found enough instances
                if len(class_detections) >= MAX_INSTANCES_PER_CLASS:
                    break
            
            # Add class ID to detections
            for det in class_detections[:MAX_INSTANCES_PER_CLASS]:
                det['class_id'] = class_id
                all_detections.append(det)
                stats['detections_per_class'][CLASS_NAMES[class_id]] += 1
        
        # Save YOLO format labels
        if all_detections:
            label_path = os.path.join(LABELS_DIR, f"{image_id}.txt")
            with open(label_path, 'w') as f:
                for det in all_detections:
                    x, y, w, h = det['bbox']
                    # Convert to YOLO format (normalized center x, y, width, height)
                    x_center = (x + w / 2) / width
                    y_center = (y + h / 2) / height
                    w_norm = w / width
                    h_norm = h / height
                    f.write(f"{det['class_id']} {x_center} {y_center} {w_norm} {h_norm}\n")
            
            stats['images_with_defects'] += 1
            stats['total_detections'] += len(all_detections)
        else:
            stats['images_without_defects'] += 1
        
        stats['processed'] += 1
        
        # Save progress every 100 images
        if stats['processed'] % 100 == 0:
            with open(STATS_FILE, 'w') as f:
                json.dump(stats, f, indent=2)
            print(f"\n📊 Progress: {stats['processed']}/{len(image_files)} - "
                  f"{stats['images_with_defects']} with defects, "
                  f"{stats['total_detections']} total detections\n")
    
    except Exception as e:
        print(f"\n❌ Failed to process {image_name}: {e}")
        stats['failed_images'].append({'file': image_name, 'error': str(e)})

# Final statistics
with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)

print("\n✅ Auto-labeling complete!")
print(f"\n📊 Final Statistics:")
print(f"  Total processed: {stats['processed']}/{stats['total_images']}")
print(f"  Images with defects: {stats['images_with_defects']} ({stats['images_with_defects']/stats['processed']*100:.1f}%)")
print(f"  Images without defects: {stats['images_without_defects']} ({stats['images_without_defects']/stats['processed']*100:.1f}%)")
print(f"  Total detections: {stats['total_detections']}")
print(f"  Failed images: {len(stats['failed_images'])}")
print(f"\n📋 Detections per class:")
for class_name, count in sorted(stats['detections_per_class'].items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        print(f"  {class_name}: {count}")

## 7. Package Results for Download

In [ ]:
# Create ZIP file with labels
OUTPUT_ZIP = '/content/sam3_auto_labels.zip'

print("Creating ZIP file...")
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all label files
    for root, dirs, files in os.walk(LABELS_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, OUTPUT_DIR)
            zipf.write(file_path, arcname)
    
    # Add statistics
    zipf.write(STATS_FILE, 'labeling_stats.json')

print(f"✅ Results packaged: {OUTPUT_ZIP}")
print(f"   Size: {os.path.getsize(OUTPUT_ZIP) / (1024*1024):.2f} MB")
print(f"\n📥 Download this file and extract to your local project")

In [ ]:
# Copy to Google Drive for persistent storage
import shutil

DRIVE_OUTPUT = '/content/drive/MyDrive/sam3_auto_labels.zip'
shutil.copy(OUTPUT_ZIP, DRIVE_OUTPUT)

print(f"✅ Results also saved to Google Drive: {DRIVE_OUTPUT}")

## Next Steps

1. **Download** `sam3_auto_labels.zip` from Google Drive
2. **Extract** labels to your local project
3. **Run merge script** to create dataset v4.0:
   ```bash
   npm run dataset:merge-v4
   ```
4. **Train YOLO v4.0** on merged dataset (target: 45-55% mAP@50)

**Expected Results**:
- 2,000-3,000 recovered images with labels
- Total dataset: 5,061-6,061 images
- Performance boost: 27.1% → 45-55% mAP@50
